# Proposal Visualizations — Home Advantage
### DSC 106 Final Project
#### Diego Menchaca, Jay Manjrekar, Kavyan Patel

---
## Section 0 — Setup

In [ ]:
# ── packages ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns
from pathlib import Path

# ── consistent style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
    'font.family'      : 'sans-serif',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
})

FIGURES = Path('figures')
FIGURES.mkdir(exist_ok=True)

In [ ]:
# ── load all clean datasets ────────────────────────────────────────────────────
CLEAN = Path('clean')

wc_matches   = pd.read_csv(CLEAN / 'wc_matches.csv',    parse_dates=['date'])
baseline     = pd.read_csv(CLEAN / 'baseline_rates.csv')
host_elo     = pd.read_csv(CLEAN / 'host_elo_delta.csv')
elo_ts       = pd.read_csv(CLEAN / 'elo_timeseries.csv')
wc_finals    = pd.read_csv(CLEAN / 'wc_finals.csv')

# ── quick sanity check ────────────────────────────────────────────────────────
for name, df in [('wc_matches', wc_matches), ('baseline', baseline),
                 ('host_elo', host_elo), ('elo_ts', elo_ts), ('wc_finals', wc_finals)]:
    print(f'{name:12s}  {df.shape[0]:>4} rows x {df.shape[1]:>2} cols')


wc_matches     964 rows x 14 cols
baseline         3 rows x  4 cols
host_elo        23 rows x 10 cols
elo_ts         249 rows x  6 cols
wc_finals       22 rows x 13 cols


---
## Visualizations

### Visualization 1

In [ ]:
# ── Viz 1: Grouped bar — win / draw / loss rates by venue context ──
VENUE_LABELS = {
    'neutral_venue': 'Neutral venue\n(all intl. matches)',
    'home_venue':    'Home venue\n(all intl. matches)',
    'wc_host_match': 'World Cup\nhost at home',
}
OUTCOME_LABELS = {'home_win': 'Win', 'draw': 'Draw', 'away_win': 'Loss'}
COLORS = {'home_win': '#264653', 'draw': '#e9c46a', 'away_win': '#e76f51'}

plot_df = baseline.copy()
plot_df['venue_label'] = plot_df['venue_type'].map(VENUE_LABELS)
venue_order = ['neutral_venue', 'home_venue', 'wc_host_match']
plot_df = plot_df.set_index('venue_type').loc[venue_order].reset_index()

outcomes = ['home_win', 'draw', 'away_win']
x = np.arange(len(plot_df))
bar_width = 0.25

fig, ax = plt.subplots(figsize=(9, 5.5))

for i, outcome in enumerate(outcomes):
    heights = plot_df[outcome] * 100
    bars = ax.bar(
        x + (i - 1) * bar_width, heights, bar_width,
        label=OUTCOME_LABELS[outcome], color=COLORS[outcome],
        edgecolor='white', linewidth=0.5,
    )
    for bar in bars:
        h = bar.get_height()
        if h > 8:
            ax.text(
                bar.get_x() + bar.get_width() / 2, h - 3,
                f'{h:.1f}%', ha='center', va='top',
                fontsize=8, color='white', fontweight='bold',
            )

ax.set_xticks(x)
ax.set_xticklabels(plot_df['venue_label'])
ax.set_ylabel('Share of matches (%)')
ax.set_ylim(0, 72)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.set_title('Home-field uplift: win rates by venue context', pad=12)
ax.legend(title='Result for home side', loc='upper left', frameon=True)

neutral_win = plot_df.loc[plot_df['venue_type'] == 'neutral_venue', 'home_win'].iloc[0] * 100
host_win    = plot_df.loc[plot_df['venue_type'] == 'wc_host_match', 'home_win'].iloc[0] * 100
gap_pp      = host_win - neutral_win

ax.annotate(
    '', xy=(2, host_win), xytext=(0, neutral_win),
    arrowprops=dict(arrowstyle='<->', color='#6c757d', lw=1.5),
)
ax.text(
    1, (neutral_win + host_win) / 2 + 2,
    f'+{gap_pp:.1f} pp\nvs neutral', ha='center', fontsize=9, color='#495057',
)

fig.tight_layout()
fig.savefig(FIGURES / 'viz1_baseline_win_rates.png', bbox_inches='tight')
plt.show()

### Visualization 2

In [ ]:
# ── Viz 2: Slopegraph — Elo-expected rank vs actual finish (all host appearances) ──
OVER   = '#2a9d8f'   # overperformed (positive delta)
UNDER  = '#e76f51'   # underperformed
MET    = '#adb5bd'   # met expectation

fig, ax = plt.subplots(figsize=(10, 11))

for _, row in host_elo.iterrows():
    delta = row['delta']
    color = OVER if delta > 0 else (UNDER if delta < 0 else MET)
    lw    = 2.2 if abs(delta) >= 7 else 1.2
    alpha = 0.95 if abs(delta) >= 7 else 0.45

    ax.plot(
        [0, 1],
        [row['elo_rank_among_participants'], row['actual_place']],
        color=color, lw=lw, alpha=alpha, zorder=2,
    )

    if abs(delta) >= 7 or row['year'] in (2014, 2002):
        mid_y = (row['elo_rank_among_participants'] + row['actual_place']) / 2
        ax.annotate(
            f"{int(row['year'])} {row['host']}",
            xy=(0.5, mid_y), fontsize=7, ha='center',
            color=color, fontweight='bold',
        )

ax.set_xlim(-0.15, 1.15)
ax.set_xticks([0, 1])
ax.set_xticklabels([
    'Elo-expected rank\n(among WC participants)',
    'Actual finish\n(lower = better)',
])
ax.set_ylabel('Finishing position (1 = champion)')
ax.invert_yaxis()
ax.set_title('World Cup hosts: pretournament Elo rank vs actual finish', pad=14)

n_positive = (host_elo['delta'] > 0).sum()
mean_delta = host_elo['delta'].mean()
ax.text(
    0.5, 0.02,
    f'{n_positive} of {len(host_elo)} hosts finished above Elo expectation'
    f'  ·  mean Δ = +{mean_delta:.1f} positions',
    transform=ax.transAxes, ha='center', fontsize=10, color='#495057',
)

legend_handles = [
    mlines.Line2D([], [], color=OVER,  lw=2, label='Overperformed (Δ > 0)'),
    mlines.Line2D([], [], color=UNDER, lw=2, label='Underperformed (Δ < 0)'),
    mlines.Line2D([], [], color=MET,   lw=2, label='Met expectation (Δ = 0)'),
]
ax.legend(handles=legend_handles, loc='upper right', frameon=True)

fig.tight_layout()
fig.savefig(FIGURES / 'viz2_host_elo_slopegraph.png', bbox_inches='tight')
plt.show()